# Django, Advanced Validation

## Introduction to Advanced Validation
---

In this lesson, you'll learn how to **implement custom validation logic** in Django forms.

Beyond basic field validators, you'll create custom validation methods, cross-field validation, and reusable validators.

1. [Validation Flow](#Validation-Flow),
    - [How Django Validates Forms](#How-Django-Validates-Forms),
    - [Where Errors Are Stored](#Where-Errors-Are-Stored),
2. [Field-Level Validation](#Field-Level-Validation),
    - [clean_fieldname() Methods](#clean_fieldname()-Methods),
    - [Accessing Other Field Values](#Accessing-Other-Field-Values),
3. [Form-Level Validation](#Form-Level-Validation),
    - [The clean() Method](#The-clean()-Method),
    - [Cross-Field Validation](#Cross-Field-Validation),
4. [Custom Validators](#Custom-Validators),
    - [Creating Reusable Validators](#Creating-Reusable-Validators),
    - [Validators with Parameters](#Validators-with-Parameters),
5. [Error Handling and Display](#Error-Handling-and-Display),
    - [Adding Custom Errors](#Adding-Custom-Errors),
    - [Non-Field Errors](#Non-Field-Errors),
    - [Displaying Errors in Templates](#Displaying-Errors-in-Templates),
6. [🧠 EXERCISE 🧠, Password Validation Form](#🧠-EXERCISE-🧠,-Password-Validation-Form),
7. [🧠 EXERCISE 🧠, Date Range Validation](#🧠-EXERCISE-🧠,-Date-Range-Validation).

<br>

## Validation Flow

---

### How Django Validates Forms

---

When you call `form.is_valid()`, Django runs validation in a specific order:

<br>

```
is_valid()
    │
    ▼
┌─────────────────────────────────────────────────┐
│  1. to_python() - Convert input to Python type  │
└─────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────┐
│  2. validate() - Run built-in field validation  │
└─────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────┐
│  3. run_validators() - Run custom validators    │
└─────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────┐
│  4. clean_<fieldname>() - Field-specific logic  │
└─────────────────────────────────────────────────┘
    │
    ▼ (after all fields)
┌─────────────────────────────────────────────────┐
│  5. clean() - Cross-field validation            │
└─────────────────────────────────────────────────┘
    │
    ▼
  Returns True/False
```

<br>

**Key insight:** Each step only runs if the previous step passed. If `to_python()` fails for a field, Django won't run `clean_<fieldname>()` for that field.

<br>

### Where Errors Are Stored

---

After validation, errors are stored in the form's `errors` attribute:

<br>

| Attribute | Description | Access |
|-----------|-------------|--------|
| `form.errors` | Dictionary of all errors | `{'email': ['Enter a valid email']}` |
| `form.errors['fieldname']` | Errors for specific field | `['Error 1', 'Error 2']` |
| `form.non_field_errors()` | Errors not tied to a field | Cross-field validation errors |

In [ ]:
# Example: Inspecting form errors

from django import forms


class SimpleForm(forms.Form):
    email = forms.EmailField()
    age = forms.IntegerField(min_value=0)


# Create form with invalid data
form = SimpleForm(data={'email': 'invalid', 'age': '-5'})

# Check validity
is_valid = form.is_valid()
print(f"Is valid: {is_valid}")

# Access all errors
print(f"All errors: {form.errors}")

# Access specific field errors
print(f"Email errors: {form.errors.get('email', [])}")
print(f"Age errors: {form.errors.get('age', [])}")

# Check for non-field errors
print(f"Non-field errors: {form.non_field_errors()}")

<br>

## Field-Level Validation

---

### clean_fieldname() Methods

---

For custom validation on a single field, define a method named `clean_<fieldname>()`:

In [ ]:
from django import forms
from django.core.exceptions import ValidationError


class RegistrationForm(forms.Form):
    """User registration form with custom validation."""
    
    username = forms.CharField(max_length=30)
    email = forms.EmailField()
    
    def clean_username(self):
        """Validate and clean the username field."""
        
        # Get the value from cleaned_data (already type-converted)
        username = self.cleaned_data['username']
        
        # Check for forbidden usernames
        forbidden = ['admin', 'root', 'superuser', 'administrator']
        if username.lower() in forbidden:
            raise ValidationError(
                f"The username '{username}' is not allowed."
            )
        
        # Check for spaces
        if ' ' in username:
            raise ValidationError(
                "Username cannot contain spaces."
            )
        
        # IMPORTANT: Always return the cleaned value
        return username.lower()  # Normalize to lowercase
    
    def clean_email(self):
        """Validate and clean the email field."""
        
        email = self.cleaned_data['email']
        
        # Block certain domains
        blocked_domains = ['tempmail.com', 'throwaway.com']
        domain = email.split('@')[1]
        
        if domain in blocked_domains:
            raise ValidationError(
                f"Registration with {domain} is not allowed."
            )
        
        return email

In [ ]:
# Test the validation

# Test forbidden username
form1 = RegistrationForm(data={'username': 'Admin', 'email': 'test@example.com'})
print(f"Admin valid: {form1.is_valid()}")
print(f"Errors: {form1.errors}")

print()

# Test valid data
form2 = RegistrationForm(data={'username': 'johndoe', 'email': 'john@example.com'})
print(f"johndoe valid: {form2.is_valid()}")
if form2.is_valid():
    print(f"Cleaned username: {form2.cleaned_data['username']}")

<br>

**Rules for `clean_<fieldname>()` methods:**

| Rule | Description |
|------|-------------|
| Access via `self.cleaned_data` | The value is already type-converted |
| Raise `ValidationError` | For invalid values |
| Return the value | Always return the (possibly modified) value |
| Return value goes to `cleaned_data` | The returned value replaces the original |

<br>

### Accessing Other Field Values

---

In `clean_<fieldname>()`, you can access other fields that have already been cleaned:

In [ ]:
from django import forms
from django.core.exceptions import ValidationError


class OrderForm(forms.Form):
    """Form where some validation depends on other fields."""
    
    product_type = forms.ChoiceField(
        choices=[('book', 'Book'), ('ebook', 'E-Book')]
    )
    quantity = forms.IntegerField(min_value=1)
    shipping_address = forms.CharField(required=False)
    
    def clean_shipping_address(self):
        """Shipping address is required for physical products."""
        
        address = self.cleaned_data.get('shipping_address')
        product_type = self.cleaned_data.get('product_type')
        
        # product_type might not exist if it failed validation
        if product_type == 'book' and not address:
            raise ValidationError(
                "Shipping address is required for physical books."
            )
        
        return address

<br>

**Caution:** Fields are cleaned in declaration order. When accessing `self.cleaned_data['other_field']`, the other field must be declared **before** the current field in the form class.

<br>

## Form-Level Validation

---

### The clean() Method

---

For validation that involves multiple fields together, override the `clean()` method:

In [ ]:
from django import forms
from django.core.exceptions import ValidationError


class PasswordChangeForm(forms.Form):
    """Form for changing password with confirmation."""
    
    old_password = forms.CharField(widget=forms.PasswordInput)
    new_password = forms.CharField(widget=forms.PasswordInput, min_length=8)
    confirm_password = forms.CharField(widget=forms.PasswordInput)
    
    def clean(self):
        """Validate that passwords match and new != old."""
        
        # Call parent clean() first
        cleaned_data = super().clean()
        
        # Get the values (may be None if field validation failed)
        old_password = cleaned_data.get('old_password')
        new_password = cleaned_data.get('new_password')
        confirm_password = cleaned_data.get('confirm_password')
        
        # Only validate if all fields passed individual validation
        if new_password and confirm_password:
            if new_password != confirm_password:
                raise ValidationError(
                    "The two password fields must match."
                )
        
        if old_password and new_password:
            if old_password == new_password:
                raise ValidationError(
                    "New password must be different from old password."
                )
        
        # Return the cleaned data dictionary
        return cleaned_data

In [ ]:
# Test password matching

form = PasswordChangeForm(data={
    'old_password': 'oldpass123',
    'new_password': 'newpass123',
    'confirm_password': 'different123'  # Doesn't match!
})

print(f"Is valid: {form.is_valid()}")
print(f"Errors: {form.errors}")
print(f"Non-field errors: {form.non_field_errors()}")

<br>

### Cross-Field Validation

---

A more complex example with date range validation:

In [ ]:
from django import forms
from django.core.exceptions import ValidationError
from datetime import date


class EventForm(forms.Form):
    """Form for creating an event with date validation."""
    
    title = forms.CharField(max_length=200)
    start_date = forms.DateField(widget=forms.DateInput(attrs={'type': 'date'}))
    end_date = forms.DateField(widget=forms.DateInput(attrs={'type': 'date'}))
    max_participants = forms.IntegerField(min_value=1)
    
    def clean(self):
        """Validate date logic across fields."""
        
        cleaned_data = super().clean()
        
        start_date = cleaned_data.get('start_date')
        end_date = cleaned_data.get('end_date')
        
        if start_date and end_date:
            # End must be after start
            if end_date < start_date:
                raise ValidationError(
                    "End date must be on or after start date."
                )
            
            # Event can't be more than 30 days
            duration = (end_date - start_date).days
            if duration > 30:
                raise ValidationError(
                    f"Event duration cannot exceed 30 days (current: {duration} days)."
                )
            
            # Can't create past events
            if start_date < date.today():
                self.add_error('start_date', "Start date cannot be in the past.")
        
        return cleaned_data

<br>

**Note the difference:**

| Method | Error Type |
|--------|------------|
| `raise ValidationError(...)` | Creates a non-field error |
| `self.add_error('fieldname', ...)` | Adds error to specific field |

<br>

## Custom Validators

---

### Creating Reusable Validators

---

For validation logic you want to reuse across multiple forms or fields, create standalone validator functions:

In [ ]:
# validators.py - Reusable validators

from django.core.exceptions import ValidationError
import re


def validate_no_special_chars(value: str) -> None:
    """Validate that value contains only letters, numbers, and underscores."""
    
    if not re.match(r'^[a-zA-Z0-9_]+$', value):
        raise ValidationError(
            "Only letters, numbers, and underscores are allowed.",
            code='invalid_characters'
        )


def validate_no_profanity(value: str) -> None:
    """Validate that value doesn't contain profanity."""
    
    # Simple example - in reality, use a proper profanity filter
    bad_words = ['spam', 'inappropriate', 'offensive']
    
    for word in bad_words:
        if word in value.lower():
            raise ValidationError(
                "This content contains inappropriate language.",
                code='profanity'
            )


def validate_isbn(value: str) -> None:
    """Validate ISBN-13 format and check digit."""
    
    # Remove hyphens
    isbn = value.replace('-', '')
    
    if len(isbn) != 13:
        raise ValidationError(
            "ISBN must be 13 digits.",
            code='invalid_length'
        )
    
    if not isbn.isdigit():
        raise ValidationError(
            "ISBN must contain only digits.",
            code='invalid_format'
        )
    
    # Validate check digit (ISBN-13 algorithm)
    total = sum(
        int(digit) * (1 if i % 2 == 0 else 3)
        for i, digit in enumerate(isbn)
    )
    
    if total % 10 != 0:
        raise ValidationError(
            "Invalid ISBN check digit.",
            code='invalid_checksum'
        )

In [ ]:
# Using validators in a form

from django import forms


class BookForm(forms.Form):
    """Book form using custom validators."""
    
    title = forms.CharField(
        max_length=200,
        validators=[validate_no_profanity]
    )
    
    isbn = forms.CharField(
        max_length=17,  # 13 digits + 4 possible hyphens
        validators=[validate_isbn]
    )
    
    author_username = forms.CharField(
        max_length=30,
        validators=[validate_no_special_chars]
    )

In [ ]:
# Test the validators

form = BookForm(data={
    'title': 'Great Book',
    'isbn': '978-0-13-468599-1',  # Valid ISBN
    'author_username': 'john_doe'
})

print(f"Valid: {form.is_valid()}")

# Test with invalid ISBN
form2 = BookForm(data={
    'title': 'Great Book',
    'isbn': '978-0-13-468599-0',  # Invalid check digit
    'author_username': 'john_doe'
})

print(f"Invalid ISBN valid: {form2.is_valid()}")
print(f"ISBN errors: {form2.errors.get('isbn', [])}")

<br>

### Validators with Parameters

---

For validators that need configuration, create a class:

In [ ]:
from django.core.exceptions import ValidationError
from django.utils.deconstruct import deconstructible


@deconstructible  # Required for migrations if used in models
class FileSizeValidator:
    """Validate that a file doesn't exceed a maximum size."""
    
    def __init__(self, max_size_mb: int = 5):
        self.max_size_mb = max_size_mb
        self.max_size_bytes = max_size_mb * 1024 * 1024
    
    def __call__(self, file):
        if file.size > self.max_size_bytes:
            raise ValidationError(
                f"File size must not exceed {self.max_size_mb}MB. "
                f"Current size: {file.size / (1024*1024):.1f}MB"
            )
    
    def __eq__(self, other):
        return (
            isinstance(other, FileSizeValidator) and
            self.max_size_mb == other.max_size_mb
        )


@deconstructible
class MinWordCountValidator:
    """Validate that text has at least a minimum number of words."""
    
    def __init__(self, min_words: int = 10):
        self.min_words = min_words
    
    def __call__(self, value: str):
        word_count = len(value.split())
        if word_count < self.min_words:
            raise ValidationError(
                f"Please write at least {self.min_words} words. "
                f"Current: {word_count} words."
            )
    
    def __eq__(self, other):
        return (
            isinstance(other, MinWordCountValidator) and
            self.min_words == other.min_words
        )

In [ ]:
# Using parameterized validators

from django import forms


class ArticleForm(forms.Form):
    """Article form with custom validators."""
    
    title = forms.CharField(max_length=200)
    
    content = forms.CharField(
        widget=forms.Textarea,
        validators=[MinWordCountValidator(min_words=50)]  # At least 50 words
    )
    
    summary = forms.CharField(
        widget=forms.Textarea(attrs={'rows': 3}),
        validators=[MinWordCountValidator(min_words=10)]  # At least 10 words
    )

In [ ]:
# Test the word count validator

form = ArticleForm(data={
    'title': 'My Article',
    'content': 'This is too short.',  # Less than 50 words
    'summary': 'Quick summary here.'  # Less than 10 words
})

print(f"Valid: {form.is_valid()}")
print(f"Content errors: {form.errors.get('content', [])}")
print(f"Summary errors: {form.errors.get('summary', [])}")

<br>

## Error Handling and Display

---

### Adding Custom Errors

---

Use `add_error()` to add errors to specific fields or the form as a whole:

In [ ]:
from django import forms
from django.core.exceptions import ValidationError


class TransferForm(forms.Form):
    """Money transfer form with multiple validation scenarios."""
    
    from_account = forms.CharField(max_length=20)
    to_account = forms.CharField(max_length=20)
    amount = forms.DecimalField(max_digits=10, decimal_places=2)
    
    def clean(self):
        cleaned_data = super().clean()
        
        from_account = cleaned_data.get('from_account')
        to_account = cleaned_data.get('to_account')
        amount = cleaned_data.get('amount')
        
        # Can't transfer to same account
        if from_account and to_account and from_account == to_account:
            self.add_error(
                'to_account',
                "Cannot transfer to the same account."
            )
        
        # Amount must be positive
        if amount is not None and amount <= 0:
            self.add_error(
                'amount',
                "Transfer amount must be positive."
            )
        
        # Check balance (simulated)
        if amount and from_account:
            balance = 100.00  # Simulated balance check
            if amount > balance:
                self.add_error(
                    'amount',
                    f"Insufficient funds. Available balance: ${balance}"
                )
        
        return cleaned_data

<br>

### Non-Field Errors

---

For errors that aren't specific to any field, use `add_error(None, ...)` or raise `ValidationError` in `clean()`:

In [ ]:
from django import forms
from django.core.exceptions import ValidationError


class CheckoutForm(forms.Form):
    """Checkout form with business rule validation."""
    
    product_id = forms.IntegerField()
    quantity = forms.IntegerField(min_value=1)
    promo_code = forms.CharField(required=False)
    
    def clean(self):
        cleaned_data = super().clean()
        
        product_id = cleaned_data.get('product_id')
        quantity = cleaned_data.get('quantity')
        promo_code = cleaned_data.get('promo_code')
        
        # Simulated business logic
        stock_available = 5  # Would come from database
        
        if quantity and quantity > stock_available:
            # Add non-field error using add_error(None, ...)
            self.add_error(
                None,
                f"Sorry, only {stock_available} items available in stock."
            )
        
        if promo_code:
            valid_codes = ['SAVE10', 'WELCOME']
            if promo_code.upper() not in valid_codes:
                self.add_error('promo_code', "Invalid promo code.")
        
        # Alternative: raise ValidationError for non-field error
        # raise ValidationError("This is a non-field error.")
        
        return cleaned_data

In [ ]:
# Test non-field errors

form = CheckoutForm(data={
    'product_id': 1,
    'quantity': 10,  # More than available
    'promo_code': 'INVALID'
})

print(f"Valid: {form.is_valid()}")
print(f"Non-field errors: {form.non_field_errors()}")
print(f"Promo code errors: {form.errors.get('promo_code', [])}")

<br>

### Displaying Errors in Templates

---

Here's how to display all types of errors in your templates:

In [ ]:
# Complete template with error handling

template_with_errors = """
{% extends 'base.html' %}

{% block content %}
<h1>{{ title }}</h1>

<form method="post" novalidate>
    {% csrf_token %}
    
    {# Display non-field errors at the top #}
    {% if form.non_field_errors %}
        <div class="alert alert-danger">
            <strong>Please correct the following errors:</strong>
            <ul>
                {% for error in form.non_field_errors %}
                    <li>{{ error }}</li>
                {% endfor %}
            </ul>
        </div>
    {% endif %}
    
    {# Display each field with its errors #}
    {% for field in form %}
        <div class="form-group {% if field.errors %}has-error{% endif %}">
            <label for="{{ field.id_for_label }}">
                {{ field.label }}
                {% if field.field.required %}<span class="required">*</span>{% endif %}
            </label>
            
            {{ field }}
            
            {% if field.help_text %}
                <small class="help-text">{{ field.help_text }}</small>
            {% endif %}
            
            {% if field.errors %}
                <ul class="error-list">
                    {% for error in field.errors %}
                        <li class="error">{{ error }}</li>
                    {% endfor %}
                </ul>
            {% endif %}
        </div>
    {% endfor %}
    
    <button type="submit">Submit</button>
</form>

<style>
.has-error input,
.has-error select,
.has-error textarea {
    border-color: #dc3545;
}

.error-list {
    list-style: none;
    padding: 0;
    margin: 5px 0;
}

.error {
    color: #dc3545;
    font-size: 0.875em;
}

.required {
    color: #dc3545;
}

.alert-danger {
    background-color: #f8d7da;
    border: 1px solid #f5c6cb;
    padding: 15px;
    margin-bottom: 20px;
    border-radius: 4px;
}
</style>
{% endblock %}
"""
print(template_with_errors)

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **Validation flow** | to_python → validate → validators → clean_field → clean |
| **clean_fieldname()** | Custom validation for single field, return cleaned value |
| **clean()** | Cross-field validation, call super().clean() first |
| **ValidationError** | Raise to indicate invalid data, include helpful message |
| **add_error()** | Add error to specific field or None for non-field |
| **Custom validators** | Reusable functions/classes, use @deconstructible for models |
| **form.errors** | Dictionary of all field errors |
| **form.non_field_errors()** | List of errors not tied to any field |

<br>

### 🧠 EXERCISE 🧠, Password Validation Form

---

Create a password change form with comprehensive validation:

1. Create a `PasswordForm` with fields:
   - `current_password`
   - `new_password`
   - `confirm_password`

2. Add field-level validation for `new_password`:
   - At least 8 characters
   - Must contain at least one uppercase letter
   - Must contain at least one digit
   - Must contain at least one special character (!@#$%^&*)

3. Add form-level validation:
   - `new_password` and `confirm_password` must match
   - `new_password` must be different from `current_password`

4. Test with various inputs

<details>
    <summary>▶️ Solution</summary>
    
```python
import re
from django import forms
from django.core.exceptions import ValidationError


class PasswordForm(forms.Form):
    """Password change form with comprehensive validation."""
    
    current_password = forms.CharField(
        widget=forms.PasswordInput,
        label='Current Password'
    )
    
    new_password = forms.CharField(
        widget=forms.PasswordInput,
        label='New Password',
        help_text='At least 8 characters with uppercase, digit, and special character.'
    )
    
    confirm_password = forms.CharField(
        widget=forms.PasswordInput,
        label='Confirm New Password'
    )
    
    def clean_new_password(self):
        """Validate password strength."""
        
        password = self.cleaned_data['new_password']
        
        # Check length
        if len(password) < 8:
            raise ValidationError(
                "Password must be at least 8 characters long."
            )
        
        # Check for uppercase
        if not re.search(r'[A-Z]', password):
            raise ValidationError(
                "Password must contain at least one uppercase letter."
            )
        
        # Check for digit
        if not re.search(r'\d', password):
            raise ValidationError(
                "Password must contain at least one digit."
            )
        
        # Check for special character
        if not re.search(r'[!@#$%^&*]', password):
            raise ValidationError(
                "Password must contain at least one special character (!@#$%^&*)."
            )
        
        return password
    
    def clean(self):
        """Cross-field validation."""
        
        cleaned_data = super().clean()
        
        current = cleaned_data.get('current_password')
        new = cleaned_data.get('new_password')
        confirm = cleaned_data.get('confirm_password')
        
        # Check passwords match
        if new and confirm and new != confirm:
            raise ValidationError(
                "The two password fields must match."
            )
        
        # Check new is different from current
        if current and new and current == new:
            self.add_error(
                'new_password',
                "New password must be different from current password."
            )
        
        return cleaned_data


# Test the form
test_cases = [
    # Too short
    {'current_password': 'old', 'new_password': 'Short1!', 'confirm_password': 'Short1!'},
    # No uppercase
    {'current_password': 'old', 'new_password': 'password1!', 'confirm_password': 'password1!'},
    # No digit
    {'current_password': 'old', 'new_password': 'Password!', 'confirm_password': 'Password!'},
    # No special char
    {'current_password': 'old', 'new_password': 'Password1', 'confirm_password': 'Password1'},
    # Don't match
    {'current_password': 'old', 'new_password': 'Password1!', 'confirm_password': 'Different1!'},
    # Valid
    {'current_password': 'old', 'new_password': 'NewPass1!', 'confirm_password': 'NewPass1!'},
]

for data in test_cases:
    form = PasswordForm(data=data)
    print(f"Valid: {form.is_valid()}, Errors: {form.errors}")
```
</details>

<br>

### 🧠 EXERCISE 🧠, Date Range Validation

---

Create a booking form with date validation:

1. Create a `BookingForm` with:
   - `guest_name` (CharField)
   - `check_in` (DateField)
   - `check_out` (DateField)
   - `room_type` (ChoiceField: 'standard', 'deluxe', 'suite')

2. Validate:
   - Check-in must be today or in the future
   - Check-out must be after check-in
   - Maximum stay is 30 days
   - Suite bookings require at least 2 nights

3. Display appropriate error messages

<details>
    <summary>▶️ Solution</summary>
    
```python
from django import forms
from django.core.exceptions import ValidationError
from datetime import date


class BookingForm(forms.Form):
    """Hotel booking form with date validation."""
    
    ROOM_CHOICES = [
        ('standard', 'Standard Room'),
        ('deluxe', 'Deluxe Room'),
        ('suite', 'Suite'),
    ]
    
    guest_name = forms.CharField(
        max_length=100,
        label='Guest Name'
    )
    
    check_in = forms.DateField(
        widget=forms.DateInput(attrs={'type': 'date'}),
        label='Check-in Date'
    )
    
    check_out = forms.DateField(
        widget=forms.DateInput(attrs={'type': 'date'}),
        label='Check-out Date'
    )
    
    room_type = forms.ChoiceField(
        choices=ROOM_CHOICES,
        label='Room Type'
    )
    
    def clean_check_in(self):
        """Validate check-in is not in the past."""
        
        check_in = self.cleaned_data['check_in']
        
        if check_in < date.today():
            raise ValidationError(
                "Check-in date cannot be in the past."
            )
        
        return check_in
    
    def clean(self):
        """Cross-field validation for dates and room type."""
        
        cleaned_data = super().clean()
        
        check_in = cleaned_data.get('check_in')
        check_out = cleaned_data.get('check_out')
        room_type = cleaned_data.get('room_type')
        
        if check_in and check_out:
            # Check-out must be after check-in
            if check_out <= check_in:
                self.add_error(
                    'check_out',
                    "Check-out must be after check-in date."
                )
                return cleaned_data
            
            # Calculate duration
            duration = (check_out - check_in).days
            
            # Maximum 30 days
            if duration > 30:
                raise ValidationError(
                    f"Maximum stay is 30 days. Selected duration: {duration} days."
                )
            
            # Suite requires minimum 2 nights
            if room_type == 'suite' and duration < 2:
                self.add_error(
                    'room_type',
                    "Suite bookings require a minimum stay of 2 nights."
                )
        
        return cleaned_data


# Test cases
from datetime import timedelta

today = date.today()

# Valid booking
form1 = BookingForm(data={
    'guest_name': 'John Doe',
    'check_in': today + timedelta(days=1),
    'check_out': today + timedelta(days=5),
    'room_type': 'standard'
})
print(f"Valid booking: {form1.is_valid()}")

# Suite with 1 night
form2 = BookingForm(data={
    'guest_name': 'Jane Doe',
    'check_in': today + timedelta(days=1),
    'check_out': today + timedelta(days=2),
    'room_type': 'suite'
})
print(f"Suite 1 night valid: {form2.is_valid()}")
print(f"Suite errors: {form2.errors}")
```
</details>

---